# Khám Phá, Đánh Giá & Làm Sạch Bảng: `job_descriptions`

Notebook này tập trung xử lý bảng dữ liệu văn bản chi tiết **`job_descriptions`** trong mô hình Silver Layer:
- Quan hệ 1-1 với bảng `jobs` thông qua khóa chính `job_id`.
- Tách riêng khỏi bảng `jobs` để tối ưu hiệu năng truy vấn bảng danh mục và phân tích.
- Làm sạch thẻ HTML, chuẩn hóa đầu dòng, nhận diện ngôn ngữ bài đăng (vi, en, vi-en) và tạo mã băm SHA-256 (text_hash) phục vụ phát hiện nội dung trùng lặp.

Các bước thực hiện:
1. Nạp dữ liệu Bronze và đồng bộ khóa ngoại `job_id` từ bảng `jobs`.
2. Đánh giá tỷ lệ khuyết thiếu, độ dài văn bản và mức độ lẫn tạp thẻ HTML.
3. Xây dựng bộ hàm làm sạch text chuyên dụng và bộ nhận diện ngôn ngữ.
4. Tạo bảng `job_descriptions` chuẩn hóa theo DDL PostgreSQL.
5. Thống kê phân bố ngôn ngữ, độ dài nội dung và xem trước kết quả.

In [1]:
import os
import re
import json
import hashlib
from collections import Counter
from datetime import datetime
import pandas as pd
import numpy as np

# Cấu hình hiển thị pandas
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)
print('Môi trường xử lý bảng job_descriptions đã sẵn sàng!')

Môi trường xử lý bảng job_descriptions đã sẵn sàng!


## 1. Nạp Dữ Liệu Nguồn & Đồng Bộ Khóa Ngoại `job_id`

In [2]:
listings_path = os.path.abspath('../../data/bronze/bronze_listings_master.json')
details_path = os.path.abspath('../../data/bronze/bronze_details_valid.json')

with open(listings_path, 'r', encoding='utf-8') as f:
    bronze_listings = json.load(f)
with open(details_path, 'r', encoding='utf-8') as f:
    bronze_details = json.load(f)

df_listings = pd.DataFrame(bronze_listings)
df_details = pd.DataFrame(bronze_details)

# Inner join theo đúng thứ tự đã tạo trong bảng 02_jobs.ipynb
df_merged = pd.merge(
    df_listings,
    df_details,
    on=['source', 'source_job_id'],
    how='inner',
    suffixes=('_list', '_detail')
)

# Gán job_id đồng bộ 100% với bảng jobs (1-indexed)
df_merged['job_id'] = df_merged.index + 1

print(f'Tổng số bản ghi đồng bộ: {len(df_merged):,} tin')
print('Danh sách các trường văn bản tiếp nhận: responsibilities_raw, requirements_raw, benefits_raw, overview_raw')

Tổng số bản ghi đồng bộ: 2,327 tin
Danh sách các trường văn bản tiếp nhận: responsibilities_raw, requirements_raw, benefits_raw, overview_raw


## 2. Kiểm Kê Chất Lượng Dữ Liệu Văn Bản (Data Quality Profiling)

In [3]:
text_fields = ['overview_raw', 'responsibilities_raw', 'requirements_raw', 'benefits_raw', 'required_skills_raw']
quality_metrics = []

for col in text_fields:
    series = df_merged[col].fillna('').astype(str).str.strip()
    empty_cnt = (series == '').sum()
    html_cnt = series.str.contains(r'<[^>]+>', regex=True).sum()
    lengths = series.apply(len)
    quality_metrics.append({
        'Trường dữ liệu': col,
        'Số lượng rỗng': empty_cnt,
        'Tỷ lệ đầy đủ (%)': round((len(series) - empty_cnt) / len(series) * 100, 2),
        'Chứa thẻ HTML': html_cnt,
        'Tỷ lệ lẫn thẻ HTML (%)': round(html_cnt / len(series) * 100, 2),
        'Độ dài ký tự trung bình': int(lengths.mean()),
        'Độ dài ký tự tối đa': int(lengths.max())
    })

df_quality = pd.DataFrame(quality_metrics)
print('--- BẢNG KIỂM KÊ CHẤT LƯỢNG CÁC TRƯỜNG VĂN BẢN ---')
print(df_quality.to_string(index=False))

--- BẢNG KIỂM KÊ CHẤT LƯỢNG CÁC TRƯỜNG VĂN BẢN ---
      Trường dữ liệu  Số lượng rỗng  Tỷ lệ đầy đủ (%)  Chứa thẻ HTML  Tỷ lệ lẫn thẻ HTML (%)  Độ dài ký tự trung bình  Độ dài ký tự tối đa
        overview_raw            994             57.28              1                    0.04                      590                 6956
responsibilities_raw              1             99.96              2                    0.09                      661                 6444
    requirements_raw              7             99.70              1                    0.04                      600                 9517
        benefits_raw              2             99.91             27                    1.16                      544                 4310
 required_skills_raw              0            100.00              0                    0.00                      111                  419


## 3. Xây Dựng Bộ Hàm Làm Sạch & Chuẩn Hóa Văn Bản

Các quy tắc làm sạch:
1. **Giải mã ký tự HTML**: Chuyển các ký tự mã hóa như `&nbsp;`, `&amp;`, `&lt;`, `&gt;`, `&quot;` về ký tự thực.
2. **Xử lý thẻ ngắt dòng**: Thay thế các thẻ `<br>`, `</p>`, `</li>`, `</div>`, `</h*>` thành ký tự xuống dòng.
3. **Lọc bỏ toàn bộ thẻ HTML còn lại**: Loại bỏ tất cả thẻ `<...>` còn sót lại.
4. **Chuẩn hóa đầu dòng (Bullet points)**: Đồng nhất các ký tự bullet (`-`, `*`, `+`, `–`, `-`) thành `- `.
5. **Loại bỏ khoảng trắng thừa**: Xóa khoảng trắng dư thừa ở đầu/cuối mỗi dòng và giảm các dòng trống liên tiếp.
6. **Nhận diện ngôn ngữ**: Xác định bài đăng là Tiếng Việt (`vi`), Tiếng Anh (`en`), hoặc song ngữ (`vi-en`).
7. **Tạo mã băm SHA-256**: Tạo `text_hash` trên toàn bộ nội dung văn bản sạch để phát hiện trùng lặp nội dung.

In [4]:
def clean_html_text(text):
    if not text or pd.isna(text):
        return ''
    s = str(text)
    # 1. Unescape HTML entities
    s = s.replace('&nbsp;', ' ').replace('&amp;', '&').replace('&lt;', '<').replace('&gt;', '>').replace('&quot;', '"')
    # 2. Thay the the ngat khoi bang xuong dong
    s = re.sub(r'<(?:br|/p|/li|/div|/h\d)[^>]*>', '\n', s, flags=re.IGNORECASE)
    # 3. Loai bo cac the HTML con lai
    s = re.sub(r'<[^>]+>', ' ', s)
    # 4. Chuan hoa cac dau gach dau dong (bullet points)
    s = re.sub(r'^[ \t]*[-\*\-\+–][ \t]*', '- ', s, flags=re.MULTILINE)
    # 5. Lam gon khoang trang giua cac tu
    lines = [re.sub(r'[ \t]+', ' ', line).strip() for line in s.split('\n')]
    # 6. Giu lai cac dong co noi dung, tranh dong trong keo dai
    compact_lines = [line for line in lines if line]
    return '\n'.join(compact_lines)

def detect_language(text):
    if not text or pd.isna(text):
        return 'vi'
    s = str(text).lower()
    vn_chars = set('àáảãạăằắẳẵặâầấẩẫậèéẻẽẹêềếểễệìíỉĩịòóỏõọôồốổỗộơờớởỡợùúủũụưừứửữựỳýỷỹỵđ')
    vn_words = ['và', 'các', 'của', 'trong', 'với', 'cho', 'khi', 'được', 'người', 'công việc', 'kinh nghiệm', 'yêu cầu', 'quyền lợi', 'kỹ năng']
    en_words = ['and', 'the', 'of', 'in', 'with', 'for', 'when', 'experience', 'requirements', 'skills', 'responsibilities', 'benefits', 'team', 'develop']
    
    vn_char_count = sum(1 for c in s if c in vn_chars)
    vn_word_hits = sum(1 for w in vn_words if re.search(r'\b' + re.escape(w) + r'\b', s))
    en_word_hits = sum(1 for w in en_words if re.search(r'\b' + re.escape(w) + r'\b', s))
    
    if vn_char_count > 10 or vn_word_hits >= 3:
        if en_word_hits >= 4 and vn_word_hits >= 3:
            return 'vi-en'
        return 'vi'
    elif en_word_hits > vn_word_hits:
        return 'en'
    return 'vi'

def compute_sha256(text):
    if not text:
        return None
    return hashlib.sha256(text.encode('utf-8')).hexdigest()

print('Bộ hàm làm sạch văn bản và nhận diện ngôn ngữ đã sẵn sàng!')

Bộ hàm làm sạch văn bản và nhận diện ngôn ngữ đã sẵn sàng!


## 4. Khởi Tạo Bảng `job_descriptions` Hoàn Chỉnh

Tiến hành làm sạch song song các cột, ghép nội dung mô tả (`overview_raw` + `responsibilities_raw`), tính toán ngôn ngữ và mã băm SHA-256.

In [5]:
desc_records = []
now_iso = datetime.now().isoformat()

for idx, row in df_merged.iterrows():
    job_id = row['job_id']
    overview = str(row.get('overview_raw', '') or '').strip()
    responsibilities = str(row.get('responsibilities_raw', '') or '').strip()
    requirements = str(row.get('requirements_raw', '') or '').strip()
    benefits = str(row.get('benefits_raw', '') or '').strip()
    
    if overview and responsibilities:
        desc_raw = f"{overview}\n\n{responsibilities}"
    elif overview:
        desc_raw = overview
    else:
        desc_raw = responsibilities
        
    desc_clean = clean_html_text(desc_raw)
    req_clean = clean_html_text(requirements)
    ben_clean = clean_html_text(benefits)
    
    full_content = f"{desc_clean}\n{req_clean}\n{ben_clean}".strip()
    lang = detect_language(full_content)
    text_hash = compute_sha256(full_content)
    
    desc_records.append({
        'job_id': job_id,
        'description_raw': desc_raw,
        'requirements_raw': requirements,
        'benefits_raw': benefits,
        'description_clean': desc_clean,
        'requirements_clean': req_clean,
        'benefits_clean': ben_clean,
        'language': lang,
        'text_hash': text_hash,
        'created_at': now_iso,
        'updated_at': now_iso
    })

df_job_descriptions = pd.DataFrame(desc_records)

print('Tạo bảng job_descriptions thành công:')
print(f'  - Tổng số bản ghi: {len(df_job_descriptions):,}')
print(f'  - Tính toàn vẹn khóa chính job_id: {df_job_descriptions["job_id"].nunique():,} / {len(df_job_descriptions)} (100.0% duy nhất)')
print(f'  - Số bản ghi có mã băm SHA-256 hợp lệ: {df_job_descriptions["text_hash"].notna().sum():,} / {len(df_job_descriptions)}')

Tạo bảng job_descriptions thành công:
  - Tổng số bản ghi: 2,327
  - Tính toàn vẹn khóa chính job_id: 2,327 / 2327 (100.0% duy nhất)
  - Số bản ghi có mã băm SHA-256 hợp lệ: 2,327 / 2327


## 5. Đánh Giá Kết Quả, Thống Kê Phân Bố & Xem Trước Bảng `job_descriptions`

In [6]:
print('--- 1. CẤU TRÚC BẢNG job_descriptions (INFO) ---')
print(df_job_descriptions.info())

print('\n--- 2. PHÂN BỐ NGÔN NGỮ BÀI ĐĂNG (language) ---')
lang_counts = df_job_descriptions['language'].value_counts()
lang_percent = df_job_descriptions['language'].value_counts(normalize=True) * 100
df_lang_stat = pd.DataFrame({'Số lượng': lang_counts, 'Tỷ lệ (%)': lang_percent.round(2)})
print(df_lang_stat.to_string())

print('\n--- 3. THỐNG KÊ SỐ LƯỢNG TỪ & ĐỘ DÀI VĂN BẢN SAU LÀM SẠCH ---')
df_len_stat = pd.DataFrame({
    'Từ Mô tả': df_job_descriptions['description_clean'].apply(lambda x: len(x.split()) if x else 0),
    'Từ Yêu cầu': df_job_descriptions['requirements_clean'].apply(lambda x: len(x.split()) if x else 0),
    'Từ Quyền lợi': df_job_descriptions['benefits_clean'].apply(lambda x: len(x.split()) if x else 0)
})
print(df_len_stat.describe().round(1).to_string())

print('\n--- 4. PHÁT HIỆN TRÙNG LẶP NỘI DUNG THỰC TẾ (text_hash) ---')
unique_hashes = df_job_descriptions['text_hash'].nunique()
duplicate_hashes = len(df_job_descriptions) - unique_hashes
print(f'Số mã băm nội dung duy nhất: {unique_hashes:,}')
print(f'Số bản ghi trùng hoàn toàn nội dung JD với tin khác: {duplicate_hashes:,} tin ({duplicate_hashes/len(df_job_descriptions)*100:.2f}%)')

print('\n--- 5. PREVIEW 5 DÒNG ĐẦU BẢNG job_descriptions (TRÍCH ĐOẠN NGẮN) ---')
preview_df = df_job_descriptions[['job_id', 'language', 'text_hash', 'description_clean', 'requirements_clean']].copy()
preview_df['description_clean'] = preview_df['description_clean'].apply(lambda x: (x[:80] + '...') if len(x) > 80 else x)
preview_df['requirements_clean'] = preview_df['requirements_clean'].apply(lambda x: (x[:80] + '...') if len(x) > 80 else x)
preview_df['text_hash'] = preview_df['text_hash'].apply(lambda x: (x[:12] + '...') if x else '')
print(preview_df.head(5).to_string(index=False))

--- 1. CẤU TRÚC BẢNG job_descriptions (INFO) ---
<class 'pandas.DataFrame'>
RangeIndex: 2327 entries, 0 to 2326
Data columns (total 11 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   job_id              2327 non-null   int64
 1   description_raw     2327 non-null   str  
 2   requirements_raw    2327 non-null   str  
 3   benefits_raw        2327 non-null   str  
 4   description_clean   2327 non-null   str  
 5   requirements_clean  2327 non-null   str  
 6   benefits_clean      2327 non-null   str  
 7   language            2327 non-null   str  
 8   text_hash           2327 non-null   str  
 9   created_at          2327 non-null   str  
 10  updated_at          2327 non-null   str  
dtypes: int64(1), str(10)
memory usage: 13.2 MB
None

--- 2. PHÂN BỐ NGÔN NGỮ BÀI ĐĂNG (language) ---
          Số lượng  Tỷ lệ (%)
language                     
vi            1733      74.47
en             328      14.10
vi-en          266    